### Autonomous Traders

An equity trading simulation with four Traders and a Researcher, powered by a set of MCP servers and their tools and resources:

1. An Accounts server.
2. Push notifications on phone, to alert when something happens
3. Market data, for live or simulated share prices
4. Fetch, to read a web page through a local headless browser
5. Tavily, for web search
6. Memory, a knowledge graph the researcher writes to and reads back

Plus resources to read each trader's account and their investment strategy.


### The architecture

Here is how the pieces fit together. Each of the four traders is an agent with its own MCP servers for accounts, push notifications and market data, and it calls a researcher agent as a tool. The researcher is itself an agent, with its own MCP servers for fetching pages, web search and memory. Orange is an agent, blue is an MCP server, and there are six MCP servers in all.

While I've given the Agents names like "Trader" and "Researcher", my motivation for this architecture was managing the context effectively with the right prompts and tools for reliable outcomes. It's key to avoid the trap of assigning Agent responsibilities just because that's how human teams are organized..

<img src="./assets/architecture.png" width="820" />

In [ ]:
from dotenv import load_dotenv, dotenv_values
from IPython.display import Markdown, display
import os

from backend.accounts import Account
from backend.mcp_servers import trader_mcp_servers, researcher_mcp_servers
from contextlib import AsyncExitStack
from agents import Runner, trace, add_trace_processor


from backend.market import get_share_price
from backend.accounts import Account
from backend.accounts_client import read_accounts_resource
from backend.reset import reset_traders
from backend.mcp_servers import trader_mcp_servers, researcher_mcp_servers
from backend.traders import get_researcher, get_researcher_tool, Trader
from backend.tracers import LogTracer


In [ ]:
load_dotenv(override=True)

In [ ]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
POLYGON_API_KEY = os.getenv("POLYGON_API_KEY")

In [ ]:
warren = Account.get("Warren")
print("Balance:", warren.balance)
print("Strategy:", warren.get_strategy())

### The MCP servers, and their tools

The trader and the researcher each get their own set of MCP servers, and `mcp_servers.py` builds them. Let's start each one and count the tools they expose.

In [ ]:
# Check how many MCP server 
# Get a LIST of servers list[MCPServerStdio]:
servers = trader_mcp_servers() + researcher_mcp_servers("Warren")

count = 0

for server in servers:
    async with server:
        tools = await server.list_tools()
        count += len(tools)
print(f"We have {len(servers)} MCP servers, and {count} tools") 

### The Researcher

The trader does not search the web itself. Instead it calls a Researcher agent as one of its tools. The researcher has its own MCP servers: Fetch to read pages, Tavily for web search, and a Memory it writes to and reads back.

Tavily offers several tools, from plain search to a heavyweight deep-research mode. We restrict its server to `tavily_search`, which keeps the researcher fast and focused. Choosing which tools an agent can see is itself context engineering.

Wrapping an agent as a tool is different from a handoff: with a tool the trader stays in control and gets the researcher's answer back, where a handoff would pass the whole conversation over.

In [ ]:
async with AsyncExitStack() as stack:
    servers = [await stack.enter_async_context(server) for server in researcher_mcp_servers("Warren")]
    researcher = await get_researcher(servers, "gpt-5.4-mini")
    with trace("Researcher"):
        result = await Runner.run(researcher, "What's the latest news on Amazon?", max_turns=30)
display(Markdown(result.final_output))